# Leaderboard Pipeline 04 — Build Reviewed Clean Dataset

This notebook combines safe automatic cleaning with human-confirmed review decisions and creates `BDC2026_clean_v1` without modifying the original dataset.

Safe automatic exclusions:

- corrupt or unreadable images
- redundant same-label exact MD5 duplicates

Human actions: `keep`, `relabel`, `exclude`, or `needs_second_review`. If a Moondream-assisted queue has been merged by `scripts/merge_moondream_human_decisions.py`, this notebook prefers that merged file. Moondream recommendations without human confirmation are never used.

The notebook starts in dry mode.


In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "leaderboard_pipeline":
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

DATA_ROOT = REPO_ROOT / "BDC2026"
CLEAN_ROOT = REPO_ROOT / "BDC2026_clean_v1"
PIPELINE_ROOT = REPO_ROOT / "leaderboard_pipeline_outputs"
PIPELINE_ROOT.mkdir(parents=True, exist_ok=True)
LABEL2ID = {"0_Recyclable": 0, "1_Electronic": 1, "2_Organic": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("CLEAN_ROOT:", CLEAN_ROOT)


In [ ]:
import shutil
import pandas as pd
from tqdm.auto import tqdm

STAGE_DIR = PIPELINE_ROOT / "04_clean_dataset"
STAGE_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = REPO_ROOT / "eda_outputs" / "notebook_pipeline" / "00_manifest" / "train_manifest.csv"
BASE_REVIEW_PATH = PIPELINE_ROOT / "02_label_review" / "ranked_review_queue.csv"
MERGED_REVIEW_PATH = PIPELINE_ROOT / "02_label_review" / "ranked_review_queue_with_moondream_human_decisions.csv"
REVIEW_PATH = MERGED_REVIEW_PATH if MERGED_REVIEW_PATH.exists() else BASE_REVIEW_PATH
EXACT_PATH = REPO_ROOT / "eda_outputs" / "notebook_pipeline" / "02_exact_duplicates" / "exact_duplicate_groups.csv"

if MANIFEST_PATH.exists():
    manifest = pd.read_csv(MANIFEST_PATH)
else:
    rows = []
    for class_name, label in LABEL2ID.items():
        for path in sorted((DATA_ROOT / "train" / class_name).glob("*")):
            if path.is_file():
                rows.append({
                    "path": str(path),
                    "relative_path": f"train/{class_name}/{path.name}",
                    "class_name": class_name,
                    "label": label,
                    "is_valid": True,
                })
    manifest = pd.DataFrame(rows)

review = pd.read_csv(REVIEW_PATH) if REVIEW_PATH.exists() else pd.DataFrame()
exact = pd.read_csv(EXACT_PATH) if EXACT_PATH.exists() else pd.DataFrame()
print("Review source:", REVIEW_PATH)
print("Manifest:", len(manifest), "Review rows:", len(review), "Exact rows:", len(exact))


In [ ]:
def original_key(path_value, class_name):
    return f"{class_name}/{Path(str(path_value)).name}"

manifest["source_path"] = manifest["path"].astype(str)
manifest["source_path_key"] = [
    original_key(path, class_name)
    for path, class_name in zip(manifest["path"], manifest["class_name"])
]
manifest["final_action"] = "keep"
manifest["final_label"] = manifest["label"].astype(int)
manifest["reason"] = "original_keep"

if "is_valid" in manifest.columns:
    bad = ~manifest["is_valid"].fillna(False)
    manifest.loc[bad, "final_action"] = "exclude"
    manifest.loc[bad, "reason"] = "corrupt_or_unreadable"

if len(exact):
    for _, group in exact.groupby("group_id"):
        group = group.sort_values("path").reset_index(drop=True)
        if not bool(group["cross_label"].iloc[0]):
            for _, row in group.iloc[1:].iterrows():
                key = original_key(row["path"], row["class_name"])
                mask = manifest["source_path_key"] == key
                manifest.loc[mask, "final_action"] = "exclude"
                manifest.loc[mask, "reason"] = "redundant_same_label_exact_duplicate"


In [ ]:
if len(review):
    reviewed = review[review["review_action"].fillna("").str.strip() != ""].copy()
    allowed = {"keep", "relabel", "exclude", "needs_second_review"}
    invalid = reviewed[~reviewed["review_action"].isin(allowed)]
    assert len(invalid) == 0, invalid[["path", "review_action"]]

    for _, row in reviewed.iterrows():
        key = original_key(row["resolved_path"], row["class_name"])
        mask = manifest["source_path_key"] == key
        action = row["review_action"]
        reason = str(row.get("review_reason", "") or "")

        if action == "keep":
            manifest.loc[mask, "final_action"] = "keep"
            manifest.loc[mask, "reason"] = reason or "manual_keep"
        elif action == "exclude":
            manifest.loc[mask, "final_action"] = "exclude"
            manifest.loc[mask, "reason"] = reason or "manual_exclude"
        elif action == "needs_second_review":
            manifest.loc[mask, "final_action"] = "pending_review"
            manifest.loc[mask, "reason"] = reason or "needs_second_review"
        elif action == "relabel":
            new_label = int(row["new_label"])
            assert new_label in ID2LABEL, row
            manifest.loc[mask, "final_action"] = "relabel"
            manifest.loc[mask, "final_label"] = new_label
            manifest.loc[mask, "reason"] = reason or "manual_relabel"

manifest["final_class_name"] = manifest["final_label"].map(ID2LABEL)
manifest["clean_relative_path"] = [
    f"train/{class_name}/{Path(path).name}"
    for class_name, path in zip(manifest["final_class_name"], manifest["source_path"])
]
manifest["clean_path_key"] = [
    f"{class_name}/{Path(path).name}"
    for class_name, path in zip(manifest["final_class_name"], manifest["source_path"])
]

display(manifest["final_action"].value_counts().to_frame("images"))
display(pd.crosstab(manifest["class_name"], manifest["final_class_name"]))


In [ ]:
pending = manifest[manifest["final_action"] == "pending_review"]
print("Pending second review:", len(pending))
if len(pending):
    display(pending[["source_path", "class_name", "final_class_name", "reason"]].head(30))

manifest.to_csv(STAGE_DIR / "cleaning_audit.csv", index=False)
manifest[manifest["final_action"] == "exclude"].to_csv(STAGE_DIR / "excluded_images.csv", index=False)
manifest[manifest["final_action"] == "relabel"].to_csv(STAGE_DIR / "relabeled_images.csv", index=False)
manifest[manifest["final_action"] == "pending_review"].to_csv(STAGE_DIR / "pending_second_review.csv", index=False)

clean_manifest = manifest[manifest["final_action"].isin(["keep", "relabel"])].copy()
clean_manifest.to_csv(STAGE_DIR / "clean_manifest.csv", index=False)
print("Clean images:", len(clean_manifest))


In [ ]:
APPLY_CLEAN_COPY = False
COPY_MODE = "symlink"  # symlink or copy
RESET_EXISTING = False

if APPLY_CLEAN_COPY:
    assert len(pending) == 0, "Resolve pending second-review rows before creating the final clean dataset."
    if CLEAN_ROOT.exists() and RESET_EXISTING:
        shutil.rmtree(CLEAN_ROOT)
    CLEAN_ROOT.mkdir(parents=True, exist_ok=True)

    collisions = []
    for _, row in tqdm(clean_manifest.iterrows(), total=len(clean_manifest), desc="Building clean train"):
        src = Path(row["source_path"])
        dst = CLEAN_ROOT / row["clean_relative_path"]
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists() or dst.is_symlink():
            if dst.resolve() != src.resolve():
                collisions.append((str(src), str(dst)))
            continue
        if COPY_MODE == "symlink":
            dst.symlink_to(src.resolve())
        else:
            shutil.copy2(src, dst)

    assert not collisions, f"Filename collisions: {collisions[:10]}"

    test_dst = CLEAN_ROOT / "test"
    if not test_dst.exists():
        if COPY_MODE == "symlink":
            test_dst.symlink_to((DATA_ROOT / "test").resolve(), target_is_directory=True)
        else:
            shutil.copytree(DATA_ROOT / "test", test_dst)

    submission_dst = CLEAN_ROOT / "submission.csv"
    if not submission_dst.exists():
        if COPY_MODE == "symlink":
            submission_dst.symlink_to((DATA_ROOT / "submission.csv").resolve())
        else:
            shutil.copy2(DATA_ROOT / "submission.csv", submission_dst)

    print("Created", CLEAN_ROOT)
else:
    print("Dry mode. Review the CSV files, then set APPLY_CLEAN_COPY=True.")


## Required before notebook 05

1. Complete human review in the base queue or Moondream human queue.
2. When using Moondream, run `scripts/merge_moondream_human_decisions.py`.
3. Resolve every row marked `needs_second_review`.
4. Rerun this notebook and check `cleaning_audit.csv`.
5. Set `APPLY_CLEAN_COPY=True` and create `BDC2026_clean_v1`.
